In [1]:
import numpy as np 
import pandas as pd  
from pathlib import Path 

CLEANED_DATAFRAME_LOCATION = Path(r'..\datasets\cleaned_dataset.csv')

cleaned_df = pd.read_csv(CLEANED_DATAFRAME_LOCATION)
cleaned_df.sample(2)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost
4955,4956,US-2023-167472-CHO-MIL-31000,06-06-2025,27-11-2028,First Class,167472,United States,Little Rock,Arkansas,72209,Chocolate,Gulf,CHO-MIL-31000,Wonka Bar - Milk Chocolate,9.75,3,6.33,3.42
9304,9305,US-2024-147760-CHO-FUD-51000,04-11-2025,28-04-2030,First Class,147760,United States,Greensboro,North Carolina,27405,Chocolate,Gulf,CHO-FUD-51000,Wonka Bar - Fudge Mallows,10.80,3,7.20,3.60


---
### Profitability Metric Calculation
---
For each product:
- Gross Margin (%)
- Profit per unit
- Total profit contribution

In [2]:
profit_df = cleaned_df.groupby(['Product ID', 'Product Name', 'Division'])[['Gross Profit', 'Sales', 'Units', 'Cost']].sum().sort_values('Gross Profit', ascending=False)
profit_df['Gross Margin (%)'] = profit_df['Gross Profit'] / profit_df['Sales']
profit_df['profit_per_unit'] = profit_df['Gross Profit'] / profit_df['Units']
profit_df['cost_per_unit'] = profit_df['Cost'] / profit_df['Units']
profit_df['profit_contribution'] = profit_df['Gross Profit'] / sum(profit_df['Gross Profit'])
profit_df['profit_percentage'] = (profit_df['Gross Profit'] / profit_df['Cost'])*100


print('Profitability Metric Calculation by each product')
profit_df.sample(2)

Profitability Metric Calculation by each product


,,,Gross Profit,Sales,Units,Cost,Gross Margin (%),profit_per_unit,cost_per_unit,profit_contribution,profit_percentage
Product ID,Product Name,Division,,,,,,,,,
SUG-SWE-91000,SweeTARTS,Sugar,28.70,61.50,41,32.80,0.466667,0.70,0.80,0.000307,87.500000
SUG-LAF-25000,Laffy Taffy,Sugar,33.48,53.73,27,20.25,0.623116,1.24,0.75,0.000358,165.333333


---
### Product Level Profitability Analysis
---
- Rank products by:
    - Gross profit
    - Gross margin
- Identify:
    - High-profit / high-margin products
    - High-sales / low-margin products
    - Low-sales / low-profit products

In [3]:
product_by_top_profit = profit_df.groupby(['Product ID', 'Product Name', 'Division'])[['Gross Profit']].sum().sort_values('Gross Profit', ascending=False)
product_by_top_margin = profit_df.groupby(['Product ID', 'Product Name', 'Division'])[['Gross Margin (%)']].sum().sort_values('Gross Margin (%)', ascending=False)

print('-'*70)
print('Top 10 Products by Gross Profit')
print(product_by_top_profit.head(10))
print('-'*70)
print('Top 10 Products by Gross Margin (%)')
print(product_by_top_margin.head(10))

----------------------------------------------------------------------
Top 10 Products by Gross Profit
                                                           Gross Profit
Product ID    Product Name                      Division               
CHO-SCR-58000 Wonka Bar -Scrumdiddlyumptious    Chocolate      19357.50
CHO-TRI-54000 Wonka Bar - Triple Dazzle Caramel Chocolate      18610.20
CHO-MIL-31000 Wonka Bar - Milk Chocolate        Chocolate      17443.37
CHO-NUT-13000 Wonka Bar - Nutty Crunch Surprise Chocolate      16819.95
CHO-FUD-51000 Wonka Bar - Fudge Mallows         Chocolate      16593.60
OTH-LIC-15000 Lickable Wallpaper                Other           3930.00
OTH-GUM-21000 Wonka Gum                         Other            310.70
SUG-EVE-47000 Everlasting Gobstopper            Sugar            104.00
OTH-KAZ-38000 Kazookles                         Other             92.75
SUG-HAI-55000 Hair Toffee                       Sugar             59.50
---------------------------------

In [6]:
high_profit_high_margin = profit_df[['profit_percentage', 'Gross Margin (%)']].sort_values(by=['profit_percentage', 'Gross Margin (%)'], ascending=[False, False])
high_profit_high_margin.rename(columns={'profit_percentage': 'High Profit', 'Gross Margin (%)': 'High Margin'}, inplace=True)

high_sales_low_margin = profit_df[['Sales', 'Gross Margin (%)']].sort_values(by=['Sales', 'Gross Margin (%)'], ascending=[False, True])
high_sales_low_margin.rename(columns={'Sales': 'High Sales', 'Gross Margin (%)': 'Low Margin'}, inplace=True)

low_sales_low_profit = profit_df[['Sales', 'profit_percentage']].sort_values(by=['Sales', 'profit_percentage'], ascending=[True, True])
low_sales_low_profit.rename(columns={'Sales': 'Low Sales', 'profit_percentage': 'Low Profit'}, inplace=True)

print('_*-'*20)
print('High-Profit / High-Margin Products: ')
print(high_profit_high_margin)
print('_*-'*20)
print('High-Sales / Low-Margin Products: ')
print(high_sales_low_margin)
print('_*-'*20)
print('Low-Sales / Low-Profit Products: ')
print(low_sales_low_profit)

_*-_*-_*-_*-_*-_*-_*-_*-_*-_*-_*-_*-_*-_*-_*-_*-_*-_*-_*-_*-
High-Profit / High-Margin Products: 
                                                           High Profit  \
Product ID    Product Name                      Division                 
SUG-EVE-47000 Everlasting Gobstopper            Sugar       400.000000   
SUG-HAI-55000 Hair Toffee                       Sugar       350.000000   
CHO-NUT-13000 Wonka Bar - Nutty Crunch Surprise Chocolate   249.000000   
CHO-SCR-58000 Wonka Bar -Scrumdiddlyumptious    Chocolate   227.272727   
CHO-FUD-51000 Wonka Bar - Fudge Mallows         Chocolate   200.000000   
CHO-TRI-54000 Wonka Bar - Triple Dazzle Caramel Chocolate   188.461538   
CHO-MIL-31000 Wonka Bar - Milk Chocolate        Chocolate   185.087719   
SUG-LAF-25000 Laffy Taffy                       Sugar       165.333333   
OTH-FIZ-56000 Fizzy Lifting Drinks              Sugar       150.000000   
OTH-GUM-21000 Wonka Gum                         Other       108.333333   
OTH-LIC-15000 